<div dir="rtl">
<h1>وزن واقعاً تغییر کرد؟</h1>
<p>درس 53 از 76 · حلقهٔ آموزش را خودمان بنویسیم · <code dir="ltr">47-loop</code></p>
<p><a target="_self" href="http://127.0.0.1:8000/part-08/chapter-01/47-loop.html">📖 بازگشت به همین درس</a></p>
<p>یک گام آموزش را بنویسید و تغییر وزن را از صرف محاسبهٔ مشتق جدا کنید.</p><p>پیش‌نیاز: forward، backward و Optimizer؛ ورودی و Target با شکل (B,T).</p>
<p>این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو Cell با برچسب TODO را خودتان کامل کنید. پیام INCOMPLETE یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p>از بالا به پایین اجرا کنید. پس از تغییر هر تابع، Cell آن و سپس Cell آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code>Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl">
<h2>قبل از اجرا، پیش‌بینی کنید</h2>
<p>اگر backward اجرا شود ولی Step اجرا نشود، کدام عددها تغییر می‌کنند؟</p>
</div>

<div dir="rtl"><p>پیش‌بینی من: …</p></div>

In [ ]:
import torch
from mini_gpt.config import ModelConfig
from mini_gpt.model import MiniGPT
torch.set_num_threads(1)
torch.manual_seed(7)
def make_model():
    torch.manual_seed(7)
    return MiniGPT(ModelConfig(12, 8, 16, 2, 1, 0.0))
x = torch.tensor([[1,2,3,4]])
y = torch.tensor([[2,3,4,5]])
model = make_model()
print('input / targets:', x.tolist(), y.tolist())
print('initial loss:', model(x,y)[1].item())

<div dir="rtl">
<h2>این بار شما کد بنویسید</h2>
<p>تابع update_step(Model, Optimizer, x, y) را بنویسید: حالت آموزش، پاک‌کردن مشتق قبلی، محاسبهٔ Loss، backward و Step. مقدار Loss پیش از تغییر وزن را به‌صورت float برگردانید.</p>
</div>

In [ ]:
def update_step(model, optimizer, x, y):
    # TODO: یک گام آموزش
    return None

In [ ]:
def test_exercise():
    candidate = make_model()
    optimizer = torch.optim.AdamW(candidate.parameters(), lr=0.01)
    before = candidate.token_embedding.weight.detach().clone()
    result = update_step(candidate, optimizer, x, y)
    if result is None:
        return False
    assert isinstance(result, float) and result > 0
    assert not torch.equal(before, candidate.token_embedding.weight)
    assert candidate.training
    reference = make_model()
    opt = torch.optim.AdamW(reference.parameters(), lr=0.01)
    for _ in range(2):
        opt.zero_grad(set_to_none=True)
        reference(x,y)[1].backward()
        opt.step()
    update_step(candidate, optimizer, x, y)
    for expected, actual in zip(reference.parameters(), candidate.parameters()):
        torch.testing.assert_close(actual, expected)
    return True
exercise_complete = test_exercise()
print('PASS' if exercise_complete else 'INCOMPLETE: update_step')

<div dir="rtl">
<h2>فقط یک عامل را تغییر دهید</h2>
<p>فقط نرخ را عوض کنید. وزن آغازین، ورودی و Gradient یکسان بمانند؛ آیا اندازهٔ تغییر هم یکسان است؟</p>
</div>

In [ ]:
for rate in (0.001, 0.01):
    trial = make_model()
    optimizer = torch.optim.SGD(trial.parameters(), lr=rate)
    before = trial.token_embedding.weight.detach().clone()
    trial(x,y)[1].backward()
    optimizer.step()
    print(rate, (trial.token_embedding.weight.detach()-before).norm().item())

<div dir="rtl">
<h2>خرابی را پیدا کنید</h2>
<p>دو بار مشتق یک عبارت را گرفته‌ایم، بی‌آنکه مشتق قبلی پاک شود. gradient_once(Parameter) را تعمیر کنید: مشتق (parameter-3)**2 را مستقل از فراخوانی‌های قبلی برگرداند و وزن را تغییر ندهد.</p>
</div>

In [ ]:
w = torch.nn.Parameter(torch.tensor(1.0))
for _ in range(2):
    ((w-3)**2).backward()
print('accumulated gradient:', w.grad.item(), 'single gradient:', -4.0)

<div dir="rtl">
<h2>اصلاح را خودتان بنویسید</h2>
<p>علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def gradient_once(parameter):
    # TODO: مشتق قبلی نباید جمع شود
    return None

In [ ]:
def test_repair():
    w = torch.nn.Parameter(torch.tensor(1.0))
    result = gradient_once(w)
    if result is None:
        return False
    assert float(result) == -4.0
    assert float(gradient_once(w)) == -4.0
    assert w.item() == 1.0
    assert float(gradient_once(torch.nn.Parameter(torch.tensor(4.0)))) == 2.0
    return True
repair_complete = test_repair()
print('PASS' if repair_complete else 'INCOMPLETE: gradient_once')

<div dir="rtl">
<h2>در Mini-GPT کجا به کار می‌آید؟</h2>
<p>مدل، MiniGPT واقعی است؛ حلقهٔ کوتاه این دفتر همان عملیات مرکزی mini_gpt/train.py را بدون گزارش و ذخیره اجرا می‌کند.</p>
</div>

<div dir="rtl">
<h2>با زبان خودتان توضیح دهید</h2>
<p>کدام آزمون نشان داد که دو گام مستقل نوشته‌اید، نه دو backward با مشتق انباشته؟</p>
</div>
<div dir="rtl"><p>پیش‌بینی و مشاهدهٔ من: …</p><p>علت خرابی و اصلاح من: …</p></div>

<div dir="rtl"><p><a target="_self" href="http://127.0.0.1:8000/part-08/chapter-01/47-loop.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/47-loop.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>